# Simulação de Fila de Atendimento com `asyncio`

Fila implementada em Python e usada numa simulação **assíncrona** de atendimento:

- **Tarefa 1:** novas pessoas chegam à fila a cada 1–3 segundos.
- **Tarefa 2:** atendentes retiram pessoas da fila; quanto mais atendentes, mais rápido o atendimento.
- **Tarefa 3:** a cada 10 segundos o sistema monitora a fila e **ajusta o número de atendentes** (adiciona se a fila passar de 10 pessoas, remove se estiver ociosa).

Um `asyncio.Lock` protege a fila contra acessos simultâneos. A simulação roda por 60 segundos e depois é encerrada.

In [1]:
import nest_asyncio
import asyncio
import random

# Habilita reentrada no loop de eventos
nest_asyncio.apply()

# Classe FilaAsync personalizada
class FilaAsync:
    def __init__(self):
        self.dados = []
        self._queue_lock = asyncio.Lock()

    async def enqueue(self, e):
        async with self._queue_lock:
            self.dados.append(e)

    async def dequeue(self):
        async with self._queue_lock:
            if self.dados:
                return self.dados.pop(0)
            return None

    def first(self):
        return self.dados[0] if self.dados else None

    def is_empty(self):
        return self.size() == 0

    def size(self):
        return len(self.dados)

# Cria a fila assíncrona
q = FilaAsync()

#determinando o número inicial de atendentes
n = 1

# Tarefa 1: Adiciona Pessoas na fila (1-3 segundos)
async def task1(q):
    while True:
        wait_time = random.uniform(1, 3)
        await asyncio.sleep(wait_time)
        await q.enqueue("p") #usando enqueue () para adicionar a fila

#tarefa 2: remove pessoas da fila
async def task2(q):
  while True:
    wait_time = random.uniform(10/n, 12/n)
    await asyncio.sleep(wait_time)
    if not q.is_empty():
      await q.dequeue() #usando dequeue () para remover da fila

#tarefa 3: Imprimir o tamanho da fila e adiciona atendentes se a fila estiver muito grande
async def task3(q):
  while True:
    wait_time = 10
    await asyncio.sleep(wait_time)
    print(f"o tamanho da fila é de {q.size()} pessoas.")
    global n
    if q.size() > 10:
      n += 1
      print("adicionando um atendente......total = ", n, "atendentes")
    if q.size() <= n and n > 1:
      n -= 1
      print("removendo um atendente......total = ", n, "atendentes")

# função principal que orquestra as tarefas
async def main(duracao=60):
    # cria as tarefas de forma paralela
    tarefas = [
        asyncio.create_task(task1(q)),
        asyncio.create_task(task2(q)),
        asyncio.create_task(task3(q)),
    ]
    # deixa a simulação rodar por um tempo e depois encerra as tarefas
    await asyncio.sleep(duracao)
    for t in tarefas:
        t.cancel()
    print(f"Simulação encerrada: {q.size()} pessoas na fila e {n} atendente(s).")

# executa a simulação por 60 segundos
await main(60)

o tamanho da fila é de 4 pessoas.


o tamanho da fila é de 8 pessoas.


o tamanho da fila é de 13 pessoas.
adicionando um atendente......total =  2 atendentes


o tamanho da fila é de 17 pessoas.
adicionando um atendente......total =  3 atendentes


o tamanho da fila é de 19 pessoas.
adicionando um atendente......total =  4 atendentes


Simulação encerrada: 21 pessoas na fila e 4 atendente(s).
